# IND320 — Project Part 2

## 1. Verify the Spark–Cassandra connection

Run the cells below in order using the project’s Python 3.12 `.venv` kernel in VS Code. Docker Desktop and the `cassandra` container must be running.

This check writes three sample rows to a dedicated `ind320_setup.spark_connection_test` table and reads them back through Spark. Repeating the check updates the same three primary keys. The sample rows are setup data, not API data.

The connector and port follow the course [Cassandra notebook](https://github.com/khliland/IND320/blob/main/D2Dbook/2_Data_sources/2_Databases/3_Cassandra.ipynb) and [Spark notebook](https://github.com/khliland/IND320/blob/main/D2Dbook/2_Data_sources/2_Databases/4_Spark.ipynb).

### 1.1 Check Python and select Java 17
Restart the notebook kernel before this cell if a Spark session is already running.

In [1]:
import os
import platform
import subprocess
import sys
from pathlib import Path

# The course setup uses Python 3.12 and Spark 3.5.1 with Java 17.
assert sys.version_info[:2] == (3, 12), "Select the project's Python 3.12 .venv kernel."
if platform.system() == "Darwin":
    os.environ["JAVA_HOME"] = subprocess.check_output(
        ["/usr/libexec/java_home", "-v", "17"], text=True
    ).strip()

# Spark's Python workers must use the same environment as this notebook.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

# Support launching the notebook from either the project root or notebooks/.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
assert (project_root / "requirements.txt").exists(), "Open this notebook inside the project."
ivy_cache = project_root / "no_sync" / "spark_ivy"
ivy_cache.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("Java home:", os.environ.get("JAVA_HOME", "not configured"))

Python: 3.12.14
Interpreter: /Users/agob/Projects/Streamlit_dashboard/.venv/bin/python
Java home: /Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home


### 1.2 Connect to Cassandra and prepare a test table
The Python driver creates the table structure. Spark will write and read the actual test rows.

In [2]:
from cassandra.cluster import Cluster

# Port 9042 is forwarded from Docker to this computer.
cluster = Cluster(["127.0.0.1"], port=9042, connect_timeout=10)
try:
    session = cluster.connect()
    version = session.execute("SELECT release_version FROM system.local").one()
    print("Cassandra connected:", version.release_version)

    # A single replica is sufficient for this local, single-container setup.
    session.execute("""
        CREATE KEYSPACE IF NOT EXISTS ind320_setup
        WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1}
    """)
    session.execute("""
        CREATE TABLE IF NOT EXISTS ind320_setup.spark_connection_test (
            run_id text,
            id int,
            area text,
            value double,
            PRIMARY KEY (run_id, id)
        )
    """)
    print("Test table ready.")
finally:
    # Release the Python driver's connections after preparing the table.
    cluster.shutdown()

Cassandra connected: 5.0.9


Test table ready.


### 1.3 Start Spark with the Cassandra connector
The first run downloads the connector and its Java dependencies. This can take a few minutes. Download messages and a native Hadoop library warning may appear.

In [3]:
from pyspark.sql import SparkSession

# Match the connector version and configuration used in the course notebook.
spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("IND320-Cassandra-Check")
    .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.5.1")
    .config("spark.jars.ivy", str(ivy_cache))
    .config("spark.cassandra.connection.host", "127.0.0.1")
    .config("spark.cassandra.connection.port", "9042")
    .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions")
    .config("spark.sql.catalog.mycatalog", "com.datastax.spark.connector.datasource.CassandraCatalog")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark connected:", spark.version)

:: loading settings :: url = jar:file:/Users/agob/Projects/Streamlit_dashboard/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/agob/Projects/Streamlit_dashboard/no_sync/spark_ivy/cache
The jars for the packages stored in: /Users/agob/Projects/Streamlit_dashboard/no_sync/spark_ivy/jars
com.datastax.spark#spark-cassandra-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-243123e9-cbb4-460f-bf87-12b7ca7495db;1.0
	confs: [default]


	found com.datastax.spark#spark-cassandra-connector_2.12;3.5.1 in central
	found com.datastax.spark#spark-cassandra-connector-driver_2.12;3.5.1 in central
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central


	found org.apache.cassandra#java-driver-core-shaded;4.18.1 in central
	found com.datastax.oss#native-protocol;1.5.1 in central
	found com.datastax.oss#java-driver-shaded-guava;25.1-jre-graal-sub-1 in central


	found com.typesafe#config;1.4.1 in central


...

	[SUCCESSFUL ] com.datastax.oss#java-driver-shaded-guava;25.1-jre-graal-sub-1!java-driver-shaded-guava.jar (229ms)
downloading https://repo1.maven.org/maven2/com/typesafe/config/1.4.1/config-1.4.1.jar ...
	[SUCCESSFUL ] com.typesafe#config;1.4.1!config.jar(bundle) (88ms)
downloading https://repo1.maven.org/maven2/org/slf4j/slf4j-api/1.7.26/slf4j-api-1.7.26.jar ...
	[SUCCESSFUL ] org.slf4j#slf4j-api;1.7.26!slf4j-api.jar (46ms)
downloading https://repo1.maven.org/maven2/io/dropwizard/metrics/metrics-core/4.1.18/metrics-core-4.1.18.jar ...
	[SUCCESSFUL ] io.dropwizard.metrics#metrics-core;4.1.18!metrics-core.jar(bundle) (59ms)


downloading https://repo1.maven.org/maven2/org/hdrhistogram/HdrHistogram/2.1.12/HdrHistogram-2.1.12.jar ...
	[SUCCESSFUL ] org.hdrhistogram#HdrHistogram;2.1.12!HdrHistogram.jar(bundle) (57ms)
downloading https://repo1.maven.org/maven2/org/reactivestreams/reactive-streams/1.0.3/reactive-streams-1.0.3.jar ...
	[SUCCESSFUL ] org.reactivestreams#reactive-streams;1.0.3!reactive-streams.jar (45ms)
downloading https://repo1.maven.org/maven2/org/apache/cassandra/java-driver-query-builder/4.18.1/java-driver-query-builder-4.18.1.jar ...
	[SUCCESSFUL ] org.apache.cassandra#java-driver-query-builder;4.18.1!java-driver-query-builder.jar(bundle) (63ms)
:: resolution report :: resolve 7504ms :: artifacts dl 1894ms
	:: modules in use:
	com.datastax.oss#java-driver-shaded-guava;25.1-jre-graal-sub-1 from central in [default]
	com.datastax.oss#native-protocol;1.5.1 from central in [default]
	com.datastax.spark#spark-cassandra-connector-driver_2.12;3.5.1 from central in [default]
	com.datastax.spark#spark

	16 artifacts copied, 0 already retrieved (17388kB/80ms)
26/10/07 16:41:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark connected: 3.5.1


### 1.4 Write sample rows using Spark

In [4]:
# Fixed primary keys make this check safe to rerun without duplicate rows.
run_id = "setup-check"
sample_rows = [(run_id, 1, "NO1", 10.0), (run_id, 2, "NO2", 20.0), (run_id, 3, "NO3", 30.0)]
test_data = spark.createDataFrame(
    sample_rows, schema="run_id string, id int, area string, value double"
)
(
    test_data.write
    .format("org.apache.spark.sql.cassandra")
    .options(keyspace="ind320_setup", table="spark_connection_test")
    .mode("append")
    .save()
)
print("Spark wrote 3 sample rows to Cassandra.")

Spark wrote 3 sample rows to Cassandra.



[Stage 0:>                                                          (0 + 2) / 2]



### 1.5 Read the rows back and verify the result

In [5]:
# Read from Cassandra, rather than checking the original in-memory DataFrame.
from pyspark.sql import functions as F

stored_data = (
    spark.read
    .format("org.apache.spark.sql.cassandra")
    .options(keyspace="ind320_setup", table="spark_connection_test")
    .load()
    .filter(F.col("run_id") == run_id)
    .select("id", "area", "value")
    .orderBy("id")
)
actual = [(row.id, row.area, row.value) for row in stored_data.collect()]
expected = [(row[1], row[2], row[3]) for row in sample_rows]
assert actual == expected, f"Read-back mismatch: {actual}"
for row in actual:
    print(row)
print("PASS: Spark wrote to Cassandra and read back all 3 rows correctly.")

(1, 'NO1', 10.0)
(2, 'NO2', 20.0)
(3, 'NO3', 30.0)
PASS: Spark wrote to Cassandra and read back all 3 rows correctly.


### 1.6 Stop the test session
This stops Spark. Cassandra remains running in Docker, and the test rows remain in the dedicated test table. To repeat the test, run the notebook from the top.

In [6]:
# Free the local Spark resources when the check is finished.
spark.stop()
print("Spark stopped. Cassandra is still running in Docker.")

Spark stopped. Cassandra is still running in Docker.


## AI usage — setup draft

Codex helped create and check this setup notebook, using the course Cassandra and Spark examples as references. Update this description as the project develops.

This notebook currently documents only the connection check. The API work, figures, project links, and required 300–500-word work log will be added during the assignment.